# qlib_validated_signal_weight_v1

SOXL qlib-native validated signal-weight strategy card: use Alpha158 plus legacy SOXL features, select the model on a validation segment with objective_score, and report the legacy comparison only on the held-out test segment.

In [1]:
from pathlib import Path
import json
import sys

import qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow.record_temp import SignalRecord, SigAnaRecord, PortAnaRecord
from qlib.contrib.strategy.signal_strategy import WeightStrategyBase
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while not (root / 'AGENTS.md').exists():
    if root.parent == root:
        raise RuntimeError('repo root not found')
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.workflow import QlibNativeExperimentSpec, run_qlib_native_experiment

provider_uri = root / 'data' / 'qlib' / 'soxl'
qlib.init(provider_uri=provider_uri.as_posix(), region='us', kernels=1)
_calendar = D.calendar(start_time='2012-01-03', end_time='2026-06-09', freq='day')

card_dir = root / 'research' / 'qlib_validated_signal_weight_v1'
spec = QlibNativeExperimentSpec(
    experiment='qlib_validated_signal_weight_v1',
    provider_uri=provider_uri,
    recorder_uri=card_dir / 'qlib_recorder',
    output_dir=card_dir,
    market='all',
    region='us',
    instrument='SOXL',
    train=('2012-01-03', '2017-12-29'),
    valid=('2018-01-02', '2018-12-31'),
    test=('2019-01-02', '2026-06-09'),
    model_selection_metric='objective_score',
)
result = run_qlib_native_experiment(spec)
summary = result.summary
legacy = summary['legacy_soxlbacktest_benchmark']
model_selection = summary['model_selection']

assert summary['feature_set']['source'] == 'Alpha158.get_feature_config+legacy_soxl_features'
assert model_selection['selection_segment'] == 'valid'
assert model_selection['report_segment'] == 'test'
assert summary['target_position_strategy']['threshold_source'] == 'train_prediction_median'
assert summary['target_position_strategy']['class'] == 'SignalWeightStrategy'

decision = '研究层面通过 legacy soxl_backtest 收益门槛；但仍需滚动验证后才可晋升生产。' if legacy['exceeds_legacy_rulebase'] else '未通过 legacy soxl_backtest 收益门槛；保留为验证集选择失败证据，不晋升。'
risk_note = '模型由 2018 validation 段选择，测试收益只用于报告；仍可能对单一年份切分敏感。'
conclusion = f"""# 中文结论

- 做了什么：使用 qlib.init、D.calendar/D.features、DatasetH/DataHandlerLP、R.start、SignalRecord、SigAnaRecord、PortAnaRecord 和 WeightStrategyBase 子类 SignalWeightStrategy 跑 SOXL 验证集选择实验。
- 切分方式：train=2012-01-03 到 2017-12-29，valid=2018-01-02 到 2018-12-31 用于模型选择，test=2019-01-02 到 2026-06-09 只用于报告和 legacy 比较。
- 模型逻辑：Alpha158 叠加旧 SOXL 特征，候选模型 {model_selection['candidate_count']} 个，按 {model_selection['selection_column']} 选中 {model_selection['selected_model']} / {model_selection['selected_implementation']}。
- 核心指标：qlib 策略 test 收益 {legacy['strategy_return_pct']:.2f}%，legacy soxl_backtest rulebase test 收益 {legacy['legacy_rulebase_return_pct']:.2f}%，超额 {legacy['excess_return_pct']:.2f}%；feature_count={summary['feature_set']['feature_count']}，recorder_id={summary['recorder_id']}。
- regime/window 差异：regime_metrics.csv、window_metrics.csv、backtest_window_metrics.csv 和 risk_analysis.csv 为 qlib recorder 派生证据；模型选择指标与 test 指标分别在 model_selection.csv 中记录。
- 是否晋升：{decision}
- 风险：{risk_note}
- 下一步：若通过，补 rolling/walk-forward 固定参数复验并同步 production paper/sim；若未通过，继续扩展 SPEC 模型或切分方案。
"""
(card_dir / 'conclusion.md').write_text(conclusion, encoding='utf-8')
display(Markdown(conclusion))
print(json.dumps({
    'recorder_id': summary['recorder_id'],
    'selected_model': model_selection,
    'target_position_strategy': summary['target_position_strategy'],
    'legacy_soxlbacktest_benchmark': legacy,
}, indent=2, ensure_ascii=False))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[44793:MainThread](2026-06-22 17:38:34,520) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[44793:MainThread](2026-06-22 17:38:34,522) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[44793:MainThread](2026-06-22 17:38:34,522) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[44793:MainThread](2026-06-22 17:38:34,527) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[44793:MainThread](2026-06-22 17:38:37,530) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[44793:MainThread](2026-06-22 17:38:37,531) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[44793:MainThread](2026-06-22 17:38:37,532) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


# 中文结论

- 做了什么：使用 qlib.init、D.calendar/D.features、DatasetH/DataHandlerLP、R.start、SignalRecord、SigAnaRecord、PortAnaRecord 和 WeightStrategyBase 子类 SignalWeightStrategy 跑 SOXL 验证集选择实验。
- 切分方式：train=2012-01-03 到 2017-12-29，valid=2018-01-02 到 2018-12-31 用于模型选择，test=2019-01-02 到 2026-06-09 只用于报告和 legacy 比较。
- 模型逻辑：Alpha158 叠加旧 SOXL 特征，候选模型 6 个，按 selection_objective_score 选中 linear_ic / qlib_ic_weighted_linear。
- 核心指标：qlib 策略 test 收益 6570.02%，legacy soxl_backtest rulebase test 收益 2148.77%，超额 4421.26%；feature_count=166，recorder_id=fa5df7f7aff44697924f641bc4589361。
- regime/window 差异：regime_metrics.csv、window_metrics.csv、backtest_window_metrics.csv 和 risk_analysis.csv 为 qlib recorder 派生证据；模型选择指标与 test 指标分别在 model_selection.csv 中记录。
- 是否晋升：研究层面通过 legacy soxl_backtest 收益门槛；但仍需滚动验证后才可晋升生产。
- 风险：模型由 2018 validation 段选择，测试收益只用于报告；仍可能对单一年份切分敏感。
- 下一步：若通过，补 rolling/walk-forward 固定参数复验并同步 production paper/sim；若未通过，继续扩展 SPEC 模型或切分方案。


{
  "recorder_id": "fa5df7f7aff44697924f641bc4589361",
  "selected_model": {
    "mode": "model_search",
    "selected_model": "linear_ic",
    "selected_implementation": "qlib_ic_weighted_linear",
    "selection_metric": "objective_score",
    "selection_segment": "valid",
    "report_segment": "test",
    "selection_column": "selection_objective_score",
    "report_column": "test_objective_score",
    "candidate_count": 6,
    "source": "qlib_dataset"
  },
  "target_position_strategy": {
    "class": "SignalWeightStrategy",
    "module_path": "src.backtest.strategy",
    "signal_source": "<PRED>",
    "threshold_source": "train_prediction_median",
    "score_threshold": -0.004851105918338165,
    "min_weight": 0.0,
    "max_weight": 1.0,
    "risk_degree": 0.95
  },
  "legacy_soxlbacktest_benchmark": {
    "benchmark_name": "legacy_soxl_backtest",
    "benchmark_source": "soxl_backtest.reference.replay_soxlbacktest_reference",
    "start": "2019-01-02",
    "end": "2026-06-09",
    "